## Final Project - AI for Quantitative Trading Strategies

In [1]:
# Pandas and Python
import pandas as pd
pd.options.display.float_format = '{:.4f}'.format
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
import numpy as np
import os
import time

# Graphic Libraries
import plotly.io as pio
pio.templates.default = "simple_white"
pio.renderers.default = "notebook"
import matplotlib.pyplot as plt
from IPython.display import clear_output

# AI and stats
import statsmodels.api as sm
import xgboost
from xgboost import XGBRegressor
import sklearn

# support functions
from support_functions import expected_returns_to_positions, analyze_expected_returns


In [2]:
# Define data path
data_path = "/Users/veli/Desktop/Misc/Python Scritps /QuantTradingStrategies/in_sample/"

# Risk free rate assumption
risk_free_rate = 0.05 # % per year
rfr_hourly = (1 + risk_free_rate)**(1 / (24*365)) - 1

# Suggested training set
start_date_train = "2023-01-24"
last_date_train = "2024-01-24"

# Suggested validation set
start_date_validate = "2024-01-25"
last_date_validate = "2024-07-24"

# Test set (Unavailable)
# start_date_test = "2024-07-25"
# last_date_test = "2025-01-24"

# Maximum number of features to use
max_nb_features = 400

# Random seed for feature selection
random_seed = 0

# Set a level of transaction costs
tc = 0.0000

# Set the percentage of engineered features to use
pct_engineered_features = 1

Load Data

In [3]:
# Main data
data = pd.read_csv(
    f"{data_path}data_in_sample.csv",
    index_col=0,
    header=[0,1],
)

# Make sure that the index is in the right format
data.index = pd.to_datetime(data.index)

# Visualize data
data

open                                                   \
                     VANRY    EGLD BTS    KEY HIPPO  FRONT    BLZ       ETH   
2023-01-24 00:00:00    NaN 44.2700 NaN    NaN   NaN    NaN 0.0720 1628.1900   
2023-01-24 01:00:00    NaN 43.9800 NaN    NaN   NaN    NaN 0.0721 1626.9000   
2023-01-24 02:00:00    NaN 43.9000 NaN    NaN   NaN    NaN 0.0715 1632.3100   
2023-01-24 03:00:00    NaN 44.3100 NaN    NaN   NaN    NaN 0.0729 1638.6400   
2023-01-24 04:00:00    NaN 44.3100 NaN    NaN   NaN    NaN 0.0725 1635.8000   
...                    ...     ...  ..    ...   ...    ...    ...       ...   
2024-07-24 19:00:00 0.1290 34.6580 NaN 0.0052   NaN 0.8868 0.1757 3416.6600   
2024-07-24 20:00:00 0.1262 34.3410 NaN 0.0052   NaN 0.8774 0.1740 3381.9400   
2024-07-24 21:00:00 0.1255 34.2090 NaN 0.0051   NaN 0.8722 0.1733 3362.2200   
2024-07-24 22:00:00 0.1257 34.4630 NaN 0.0051   NaN 0.8730 0.1733 3374.6900   
2024-07-24 23:00:00 0.1238 33.7750 NaN 0.0051   NaN 0.8570 0.1712 3335.0000   

                                  ...  return                                 \
                    USUAL   MEME  ...     CKB    ATOM AUCTION    ARKM POPCAT   
2023-01-24 00:00:00   NaN    NaN  ...     NaN -0.0041     NaN     NaN    NaN   
2023-01-24 01:00:00   NaN    NaN  ...     NaN  0.0023     NaN     NaN    NaN   
2023-01-24 02:00:00   NaN    NaN  ...     NaN  0.0088     NaN     NaN    NaN   
2023-01-24 03:00:00   NaN    NaN  ...     NaN  0.0051     NaN     NaN    NaN   
2023-01-24 04:00:00   NaN    NaN  ...     NaN  0.0064     NaN     NaN    NaN   
...                   ...    ...  ...     ...     ...     ...     ...    ...   
2024-07-24 19:00:00   NaN 0.0159  ... -0.0175 -0.0100 -0.0077 -0.0142    NaN   
2024-07-24 20:00:00   NaN 0.0157  ...  0.0016 -0.0036 -0.0081 -0.0106    NaN   
2024-07-24 21:00:00   NaN 0.0156  ...  0.0174  0.0021 -0.0056  0.0020    NaN   
2024-07-24 22:00:00   NaN 0.0157  ... -0.0107 -0.0150 -0.0154 -0.0087    NaN   
2024-07-24 23:00:00   NaN 0.0154  ... -0.0125 -0.0055 -0.0162 -0.0143    NaN   

                                                  
                    PNUT SPX     AXS     YFI MDT  
2023-01-24 00:00:00  NaN NaN -0.0132 -0.0018 NaN  
2023-01-24 01:00:00  NaN NaN -0.0254  0.0024 NaN  
2023-01-24 02:00:00  NaN NaN  0.0076  0.0141 NaN  
2023-01-24 03:00:00  NaN NaN  0.0000 -0.0003 NaN  
2023-01-24 04:00:00  NaN NaN  0.0144  0.0061 NaN  
...                  ...  ..     ...     ...  ..  
2024-07-24 19:00:00  NaN NaN -0.0144 -0.0067 NaN  
2024-07-24 20:00:00  NaN NaN -0.0025 -0.0005 NaN  
2024-07-24 21:00:00  NaN NaN  0.0017  0.0031 NaN  
2024-07-24 22:00:00  NaN NaN -0.0141 -0.0106 NaN  
2024-07-24 23:00:00  NaN NaN -0.0069 -0.0067 NaN  

[13152 rows x 4521 columns]

In [4]:
# Check what are the fields available
data.columns.get_level_values(0).drop_duplicates()

Index(['open', 'high', 'low', 'close', 'nb_trades', 'volume_coin',
       'volume_usd', 'open_interest', 'open_interest_value', 'funding_rate',
       'return'],
      dtype='object')

In [5]:
import psutil
print(psutil.virtual_memory())

svmem(total=17179869184, available=8358821888, percent=51.3, used=6221758464, free=4408180736, active=4134092800, inactive=3757965312, wired=2087665664)


## Features Engineering

We construct a rich set of **economically motivated features** designed to capture **short-horizon price formation**, **liquidity conditions**, **crowded positioning**, **volatility regimes**, and **trend dynamics** at the hourly frequency.  
All features are computed **asset-by-asset** and later standardized **cross-sectionally**, ensuring that the model learns **relative signals** rather than absolute scale effects.

Let $O_t, H_t, L_t, C_t$ denote the open, high, low, and close prices at time $t$, and let $r_t$ denote the one-hour return.

### **1. Candle Geometry Features**

These features summarize **intra-period price action** using candlestick geometry, capturing order-flow imbalance and short-term pressure.

**Candle body**
$$
\text{Body}_t = C_t - O_t
$$

**Candle range**
$$
\text{Range}_t = H_t - L_t
$$

**Directional candle indicator**
$$
\text{candle\_way}_t = C_t - O_t
$$

**Filling ratio (close location within the range)**
$$
\text{filling}_t = \frac{C_t - O_t}{H_t - L_t}
$$

**Amplitude (normalized range)**
$$
\text{amplitude}_t = \frac{H_t - L_t}{C_t}
$$

**Internal Bar Strength (IBS)**
$$
\text{IBS}_t = \frac{C_t - L_t}{H_t - L_t}
$$

These variables proxy **aggressive buying/selling**, **closing pressure**, and **local imbalance** within each bar.

### **2. Liquidity and Market Impact Measures**

We incorporate classic microstructure proxies that measure how costly it is to trade.

**Amihud illiquidity**
$$
\text{Amihud}_t = \frac{|r_t|}{\text{Volume}^{USD}_t}
$$

Rolling averages over horizons $w \in \{24,72\}$:
$$
\text{Amihud}_t^{(w)} =
\frac{\mathbb{E}_w[|r_t|]}{\mathbb{E}_w[\text{Volume}^{USD}_t]}
$$

**Kyle’s lambda**
$$
\lambda_t = \frac{r_t}{\text{Volume}^{USD}_t}
$$

This captures **price impact per unit of traded volume**, with rolling versions estimating persistent market impact.

### **3. Crowdedness and Leverage Pressure**

To detect **fragile market states**, we combine **funding rates** with **open interest**:

$$
\text{Crowdedness}_t = \text{Funding}_t \times \text{OI}^{USD}_t
$$

High values indicate **leveraged positioning**, where small shocks may trigger forced liquidations.  
Rolling means isolate **persistent crowding** rather than transient spikes.

### **4. Rolling Candle Aggregates**

We compute rolling statistics of candle metrics over multiple horizons $w \in \{6,24,72\}$.

These include:
- Mean candle direction
- Volatility of candle direction
- Average filling ratio

They summarize **recent price pressure consistency** and **trend persistence**.

### **5. Volume Surprise and Trade Size**

**Volume z-score**
$$
\text{VolumeZ}_t^{(w)} =
\frac{\text{Volume}_t - \mu_w(\text{Volume})}{\sigma_w(\text{Volume})}
$$

This captures **abnormal trading activity**, often associated with information arrival.

**Average trade size**
$$
\text{AvgTradeUSD}_t =
\frac{\text{Volume}^{USD}_t}{\text{Number of Trades}_t}
$$

Large values may signal **institutional participation**.

### **6. Price–Volume Interaction**

$$
\text{Return} \times \text{Volume}_t^{(w)} =
\mathbb{E}_w[r_t] \cdot \mathbb{E}_w[\text{Volume}_t]
$$

This interaction highlights **directional moves confirmed by volume**, which tend to be more informative.

### **7. Volatility Estimators**

We use multiple **range-based volatility estimators**, which are more efficient than close-to-close volatility at high frequency.

**Parkinson volatility**
$$
\sigma^{P}_t =
\sqrt{
\mathbb{E}_w\!\left[
\ln^2\!\left(\frac{H_t}{L_t}\right)
\right]
}
$$

**Yang–Zhang volatility**
$$
\sigma^{YZ}_t =
\sqrt{
\operatorname{Var}_w\!\left(\ln\frac{O_t}{C_{t-1}}\right)
+
\operatorname{Var}_w\!\left(\ln\frac{C_t}{O_t}\right)
+
\mathbb{E}_w[\text{RS}_t]
}
$$

This estimator is robust to **opening jumps** and **intraday drift**.

**Rogers–Satchell volatility**
$$
\sigma^{RS}_t =
\sqrt{
\mathbb{E}_w\!\left[
\ln\!\left(\frac{H_t}{C_t}\right)\ln\!\left(\frac{H_t}{O_t}\right)
+
\ln\!\left(\frac{L_t}{C_t}\right)\ln\!\left(\frac{L_t}{O_t}\right)
\right]
}
$$

### **8. Volatility Regimes and Volatility-of-Volatility**

To detect regime shifts, we compute **ratios of short- to long-horizon volatility**:
$$
\text{VolRegime}_t = \frac{\sigma_{t}^{\text{short}}}{\sigma_{t}^{\text{long}}}
$$

We also measure **volatility of volatility**:
$$
\text{VoV}_t = \operatorname{Std}_w(\sigma_t)
$$

High values signal **unstable market conditions**.

### **9. Asymmetric Risk Measures**

**Downside semi-volatility**
$$
\sigma^-_t =
\sqrt{
\operatorname{Var}_w(r_t \mid r_t < 0)
}
$$

**Downside-to-upside volatility ratio**
$$
\frac{\sigma^-_t}{\sigma^+_t}
$$

This captures **crash-risk asymmetry**, which is particularly relevant in crypto markets.

### **10. Trend Estimation**

Local trends are estimated using rolling linear regressions:
$$
\hat{\beta}_w =
\frac{
\sum_{i=1}^{w} (p_i - \bar p)(t_i - \bar t)
}{
\sum_{i=1}^{w} (t_i - \bar t)^2
}
$$

These slopes measure **short-term momentum strength**.

### **11. Adaptive Moving Average Deviations (KAMA)**

We compute distances to adaptive moving averages:
$$
\text{KAMA\_dist}_t =
\frac{C_t - \text{KAMA}_t}{C_t}
$$

These features capture **mean reversion versus trend continuation**, depending on the prevailing regime.

In [ ]:
cool_features = pd.DataFrame(
    np.nan,
    index=data["return"].stack().index,
    columns=[]
)

open_  = data["open"]
high   = data["high"]
low    = data["low"]
close  = data["close"]
ret = data["return"]

# candle features
body = close - open_
range_ = (high - low).replace(0, np.nan)

# base candle features
cool_features["candle_way"] = body.stack()
cool_features["filling"] = (body / range_).stack()
cool_features["amplitude"] = (range_ / close).stack()
cool_features["range_rel"] = (range_ / close).stack()
cool_features["ibs"] = ((close - low) / range_).stack()

# Amihud illiquidity

vol_usd = data["volume_usd"].replace(0, np.nan)

cool_features["amihud_illiq"] = (ret.abs() / vol_usd).stack()

for w in [24, 72]:
    cool_features[f"amihud_illiq_{w}"] = (
        ret.abs().rolling(w).mean() /
        vol_usd.rolling(w).mean()
    ).stack()

# Kyle lambda

cool_features["kyle_lambda"] = (ret / vol_usd).stack()

for w in [24, 72]:
    cool_features[f"kyle_lambda_{w}"] = (
        ret.rolling(w).mean() /
        vol_usd.rolling(w).mean()
    ).stack()

# crowdedness: high funding + large OI → crowded long leverage → fragile state

fund = data["funding_rate"]
oi_val = data["open_interest_value"]

cool_features["funding_x_oi"] = (fund * oi_val).stack()

for w in [24]:
    cool_features[f"funding_x_oi_{w}"] = (
        fund.rolling(w).mean() *
        oi_val.rolling(w).mean()
    ).stack()

# rolling candle aggregates
for w in [6, 24, 72]:
    cool_features[f"candle_way_mean_{w}"] = body.rolling(w).mean().stack()
    cool_features[f"candle_way_std_{w}"] = body.rolling(w).std().stack()
    cool_features[f"filling_mean_{w}"] = (body / range_).rolling(w).mean().stack()

# volume surprise

for w in [24, 72]:
    vol = data["volume_usd"]
    cool_features[f"volume_z_{w}"] = (
        (vol - vol.rolling(w).mean()) /
        vol.rolling(w).std()
    ).stack()

# avg trade size

trades = data["nb_trades"].replace(0, np.nan)

cool_features["avg_trade_usd"] = (
    data["volume_usd"] / trades
).stack()

# price x volume interaction

for w in [24]:
    cool_features[f"return_x_volume_{w}"] = (
        ret.rolling(w).mean() *
        data["volume_usd"].rolling(w).mean()
    ).stack()

# volatility features

log_hl = np.log(high / low)

# Parkinson volatility
for w in [6, 24, 72, 120]:
    pv = np.sqrt((log_hl ** 2).rolling(w).mean())
    cool_features[f"parkinson_vol_{w}"] = pv.stack()

# volatility regime ratios
cool_features["vol_regime_24_120"] = (
    np.sqrt((log_hl ** 2).rolling(24).mean()) /
    np.sqrt((log_hl ** 2).rolling(120).mean())
).stack()

cool_features["vol_regime_6_72"] = (
    np.sqrt((log_hl ** 2).rolling(6).mean()) /
    np.sqrt((log_hl ** 2).rolling(72).mean())
).stack()

# Yang–Zhang volatility
log_ho = np.log(high / open_)
log_lo = np.log(low / open_)
log_co = np.log(close / open_)
log_oc = np.log(open_ / close.shift(1))
rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

for w in [6, 24, 72, 120]:
    yz = np.sqrt(
        log_oc.rolling(w).var()
        + log_co.rolling(w).var()
        + rs.rolling(w).mean()
    )
    cool_features[f"yang_zhang_vol_{w}"] = yz.stack()

# Rogers–Satchell volatility

log_hc = np.log(high / close)
log_ho = np.log(high / open_)
log_lc = np.log(low / close)
log_lo = np.log(low / open_)

rs_term = log_hc * log_ho + log_lc * log_lo

for w in [6, 24, 72]:
    rs_vol = np.sqrt(rs_term.rolling(w).mean())
    cool_features[f"rogers_satchell_vol_{w}"] = rs_vol.stack()

# vol of vol (VoV)

for w in [6, 24, 72]:
    vol = np.sqrt((log_hl ** 2).rolling(w).mean())
    cool_features[f"vol_of_vol_{w}"] = vol.rolling(w).std().stack()

# downside semi-volatility

for w in [6, 24, 72]:
    downside = ret.where(ret < 0)
    cool_features[f"downside_vol_{w}"] = downside.rolling(w).std().stack()

# upside/downside ratio

upside_std = ret.where(ret > 0).rolling(24).std().replace(0, np.nan)

cool_features["down_up_vol_ratio_24"] = (
    downside.rolling(24).std() / upside_std
).stack()

# trend features (multi-horizon)

def linear_slope(x):
    t = np.arange(len(x))
    t = t - t.mean()
    return (x @ t) / (t @ t)

# Slopes
for w in [6, 12, 24]:
    cool_features[f"trend_slope_{w}h"] = (
        close.rolling(w).apply(linear_slope, raw=True)
    ).stack()

# KAMA distances (short & medium)
for span in [10, 20, 50, 120]:
    kama = close.ewm(span=span, adjust=False).mean()
    cool_features[f"kama_distance_{span}"] = ((close - kama) / close).stack()

print("Cool features shape:", cool_features.shape)
cool_features.head()

In [ ]:
simple_features = pd.DataFrame(
    np.nan,
    index=data["return"].stack().index,
    columns=[],
)

# Iterate over some interesting fields to create features
for field in [
    "return", 
    "close",
    "nb_trades", 
    "volume_usd", 
    "funding_rate", 
    "open_interest_value"
]:
    print(f"Make features using raw field {field}")
    
    # Extract raw data
    raw_data = data[field]

    # Create raw feature
    for feature_style in [
        "level",
        "delta_1",
        "delta_6",
        "shift_1",
        "shift_3",
        "shift_6",
        "shift_24",
        "mean_6",
        "mean_24",
        "mean_120",
        "std_6",
        "std_24",
        "std_120",
        "skew_6",
        "skew_24",
        "skew_120",
        "kurt_6",
        "kurt_24",
        "kurt_120",
    ]:

        # Level
        if feature_style == "level":
            raw_feature = raw_data

        # Delta 1
        if feature_style == "delta_1":
            raw_feature = raw_data.pct_change(1, fill_method=None)

        # Delta 6
        if feature_style == "delta_6":
            raw_feature = raw_data.pct_change(6, fill_method=None)

        # Shift 1
        if feature_style == "shift_1":
            raw_feature = raw_data.shift(1)

        # Shift 3
        if feature_style == "shift_3":
            raw_feature = raw_data.shift(3)

        # Shift 6
        if feature_style == "shift_6":
            raw_feature = raw_data.shift(6)

        # Shift 24
        if feature_style == "shift_24":
            raw_feature = raw_data.shift(24)

        # Mean 6
        if feature_style == "mean_6":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(6,1).mean()

        # Mean 24
        if feature_style == "mean_24":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(24,1).mean()

        # Mean 120
        if feature_style == "mean_120":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(120,1).mean()

        # Std 6
        if feature_style == "std_6":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(6,2).std()

        # Std 24
        if feature_style == "std_24":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(24,2).std()

        # Std 120
        if feature_style == "std_120":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(120,2).std()

        # Skew 6
        if feature_style == "skew_6":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(6,3).skew()

        # Skew 24
        if feature_style == "skew_24":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(24,3).skew()

        # Skew 120
        if feature_style == "skew_120":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(120,3).skew()

        # Kurtosis 6
        if feature_style == "kurt_6":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(6,4).kurt()

        # Kurtosis 24
        if feature_style == "kurt_24":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(24,4).kurt()

        # Kurtosis 120
        if feature_style == "kurt_120":
            raw_feature = raw_data.dropna(axis=1, how="all").rolling(120,4).kurt()

        # Stack and store feature
        raw_feature = raw_feature.stack().reindex(simple_features.index)
        simple_features[f"{field}_{feature_style}"] = raw_feature

<br> 
Loading features
<br> 
<br> 

In [ ]:
## takes around 9 min with all features
# Load engineered features
engineered_features = pd.DataFrame(
    np.nan,
    index=data["return"].stack().index,
    columns=[],
)

for dirpath, dirnames, filenames in os.walk(data_path):

    # Only keep standard features for the moment
    filenames = [i for i in filenames if "feature" in i]
   
    # Select a random list of features (always the same)
    np.random.seed(random_seed)
    subset_filenames = np.random.choice(
        a=filenames,
        size=min(int(max_nb_features*pct_engineered_features), len(filenames)),
        replace=False,
    )    
    for filename in subset_filenames:
          
        print(f"Loading {filename}")
        
        # Load feature
        feature = pd.read_csv(
            f"{data_path}{filename}",
            index_col=0,
            header=[0],
        )  
        
        # Make sure that the index is in the right format
        feature.index = pd.to_datetime(feature.index)
        
        # Store in the feature dict
        engineered_features[filename.replace(".csv", "")] = feature.stack().reindex(
            engineered_features.index)

In [ ]:
print(engineered_features.shape)
engineered_features.head()

<br> 
Putting all features together
<br> 
<br> 

In [ ]:
USE_COOL_FEATURES = True 

features = pd.DataFrame(
    np.nan,
    index=data["return"].stack().index,
    columns=[]
)

# optionally include cool features

if USE_COOL_FEATURES:
    for feature_name in cool_features.columns:
        features[feature_name] = cool_features[feature_name]
    n_c = len(cool_features.columns)
else:
    n_c = 0

# remaining feature budget
remaining_features = max_nb_features - n_c
remaining_features = max(remaining_features, 0)

# split remaining budget
n_engineered = int(remaining_features * pct_engineered_features)
n_simple = remaining_features - n_engineered

# sample simple features

np.random.seed(random_seed)

subset_simple_features = np.random.choice(
    a=list(simple_features.columns),
    size=min(n_simple, len(simple_features.columns)),
    replace=False,
)

for feature_name in subset_simple_features:
    features[feature_name] = simple_features[feature_name]

# sample engineered features

subset_engineered_features = np.random.choice(
    a=list(engineered_features.columns),
    size=min(n_engineered, len(engineered_features.columns)),
    replace=False,
)

for feature_name in subset_engineered_features:
    features[feature_name] = engineered_features[feature_name]

print("Cool features?:", USE_COOL_FEATURES)
print("Total features used:", features.shape[1])
print("Feature matrix shape:", features.shape)

<h4>Label Definition</h4>

<br> 

In [ ]:
import gc

if 'cool_features' in locals(): del cool_features
if 'simple_features' in locals(): del simple_features
if 'engineered_features' in locals(): del engineered_features

returns_matrix = data["return"].copy() # Keep only what we need for the label
del data 

gc.collect()

label_all = (
    returns_matrix
    .shift(-1)
    .stack()
    .dropna()
)

print("Label generation complete.")

## Features Selection

In this step, we perform an explicit feature screening procedure based on the **cross-sectional predictive relationship** between each candidate feature and next-hour returns. The objective is to aggressively reduce dimensionality while retaining features that exhibit stable and economically meaningful signal across assets and time. All computations are performed using only the training sample, and each feature is evaluated independently in a purely cross-sectional setting.

Feature selection is conducted exclusively on the training window defined by
$$
t \in [t_{\text{train,start}},\, t_{\text{train,end}}],
$$
ensuring that no information from the validation or test periods enters the screening process. Let $t_{\text{raw}}$ denote the timestamp associated with each stacked observation; the training mask simply restricts attention to observations whose timestamps fall within this interval.

For each feature, we apply **cross-sectional rank normalization** independently at each time $t$. Raw feature values are first converted into percentile ranks across assets,
$$
\tilde X_{t,i} = \operatorname{rank}_i(X_{t,i}),
$$
and then standardized cross-sectionally,
$$
Z_{t,i} =
\frac{\tilde X_{t,i} - \mu_t(\tilde X)}{\sigma_t(\tilde X)},
$$
where $\mu_t(\cdot)$ and $\sigma_t(\cdot)$ denote the cross-sectional mean and standard deviation at time $t$. This transformation removes scale effects, mitigates the influence of outliers, and ensures that each signal reflects **relative positioning within the cross-section** rather than absolute magnitudes.

To evaluate predictive content, we compute a time series of **Information Coefficients (ICs)** for each feature. At each time $t$, the IC measures the cross-sectional association between the normalized feature and next-hour returns. We compute both Pearson and Spearman versions:
$$
\text{IC}^{(P)}_t = \operatorname{corr}_i(Z_{t,i},\, y_{t+1,i}),
$$
$$
\text{IC}^{(S)}_t = \operatorname{corr}_i\!\left(\operatorname{rank}(Z_{t,i}),\, \operatorname{rank}(y_{t+1,i})\right).
$$
ICs are only computed when the cross-section contains at least 50 valid observations and exhibits non-zero variance, ensuring statistical reliability.

Each feature’s IC series is then summarized over time. We compute the mean ICs,
$$
\mu^{(P)} = \mathbb{E}[\text{IC}^{(P)}_t], \qquad
\mu^{(S)} = \mathbb{E}[\text{IC}^{(S)}_t],
$$
as well as their corresponding information ratios,
$$
\text{ICIR}^{(P)} = \frac{\mu^{(P)}}{\sigma(\text{IC}^{(P)})}, \qquad
\text{ICIR}^{(S)} = \frac{\mu^{(S)}}{\sigma(\text{IC}^{(S)})}.
$$
To enforce robustness, the final feature score is defined conservatively as
$$
\text{Score} = \min\!\left(|\mu^{(P)}|,\; |\mu^{(S)}|\right),
$$
which favors features that are consistently predictive under both linear and rank-based dependence structures.

All features are ranked in descending order according to this score, and we retain the top $K$ features.

**NOTE:** This feature screening step does introduce a degree of **selection bias**, as the same training window is reused across multiple walk-forward folds. As a result, validation Sharpe ratios may be mildly inflated relative to a fully nested feature-selection procedure. This approach is nonetheless adopted deliberately: performing IC-based screening inside every walk-forward fold would be computationally prohibitive given the size of the feature universe, and cross-sectional ICs are known to be relatively stable over time. Importantly, no information from the validation or test periods is used during feature ranking, and all reported performance metrics should be interpreted with this selection bias in mind.

In [ ]:
import gc

# 1. Calculate the intersection (Metadata only, very cheap)
common_idx = features.index.intersection(label_all.index)

print(f"Aligning features. Original shape: {features.shape}")

# 2. CRITICAL FIX: Drop unwanted rows in-place instead of copying to a new variable
# Identify rows in 'features' that are NOT in the common index
idx_to_drop = features.index.difference(common_idx)

# Remove them directly from memory
features.drop(index=idx_to_drop, inplace=True)

# Sort in-place (no copy)
features.sort_index(inplace=True)

# 3. Create the 'features_raw' reference 
# This is just a pointer to the same data, costing 0 RAM
features_raw = features 

# 4. Handle the label (it is small, so standard .loc is safe)
label_raw = label_all.loc[common_idx].sort_index()

# 5. Cleanup
del idx_to_drop
gc.collect()

print(f"Alignment complete. New shape: {features_raw.shape}")

In [ ]:
# build TRAIN mask on label_raw index (NOT label_all)
t_raw = label_raw.index.get_level_values(0)

train_mask = (
    (t_raw >= pd.to_datetime(start_date_train)) &
    (t_raw <= pd.to_datetime(last_date_train))
)

In [ ]:
from scipy.stats import spearmanr

def cs_rank_zscore(df_wide: pd.DataFrame) -> pd.DataFrame:
    x = df_wide.rank(axis=1, pct=True)
    x = x.sub(x.mean(axis=1), axis=0)
    x = x.div(x.std(axis=1).replace(0, 1), axis=0)
    return x

def compute_ic_series(
    x_wide: pd.DataFrame,
    y_wide: pd.DataFrame,
    method: str = "pearson",
    min_cs_obs: int = 50,
) -> pd.Series:
    """
    x_wide, y_wide: time x asset aligned
    returns: IC_t per time
    """
    ic_list = []
    idx = []

    common_t = x_wide.index.intersection(y_wide.index)

    for t in common_t:
        x = x_wide.loc[t].values
        y = y_wide.loc[t].values

        # finite observations only
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < min_cs_obs:
            continue

        x_m = x[m]
        y_m = y[m]

        # skip zero-variance cross-sections
        if np.std(x_m) == 0 or np.std(y_m) == 0:
            continue

        if method == "pearson":
            ic = np.corrcoef(x_m, y_m)[0, 1]
        elif method == "spearman":
            ic = spearmanr(x_m, y_m).correlation
        else:
            raise ValueError("method must be 'pearson' or 'spearman'")

        if np.isfinite(ic):
            ic_list.append(ic)
            idx.append(t)

    return pd.Series(ic_list, index=pd.to_datetime(idx), name=f"ic_{method}")

In [ ]:
# build y as time x asset for training only (next-hour return) (30-40 min)

y_train_wide = label_raw.loc[train_mask].unstack()

scores = []

for j, fname in enumerate(features_raw.columns):
    if j % 5 == 0:
        print(f"screening feature {j}/{features_raw.shape[1]} ...")

    x_wide = features_raw.loc[train_mask, fname].unstack()
    x_norm = cs_rank_zscore(x_wide)

    # align
    common_t = x_norm.index.intersection(y_train_wide.index)
    x_norm = x_norm.loc[common_t]
    y_wide = y_train_wide.loc[common_t]

    # IC series
    ic_p = compute_ic_series(x_norm, y_wide, method="pearson")
    ic_s = compute_ic_series(x_norm, y_wide, method="spearman")

    # summary stats
    mu_p, sd_p = ic_p.mean(), ic_p.std(ddof=1)
    mu_s, sd_s = ic_s.mean(), ic_s.std(ddof=1)

    # ICIR (information ratio of IC)
    icir_p = mu_p / (sd_p + 1e-12)
    icir_s = mu_s / (sd_s + 1e-12)

    # robust combined score: require both to be strong
    score = min(abs(mu_p), abs(mu_s))

    scores.append({
        "feature": fname,
        "ic_mean_pearson": mu_p,
        "ic_mean_spearman": mu_s,
        "icir_pearson": icir_p,
        "icir_spearman": icir_s,
        "score": score,
    })

scores_df = pd.DataFrame(scores).set_index("feature")
scores_df = scores_df.sort_values("score", ascending=False)

scores_df.head(20)

In [ ]:
K = 150 # change as needed (# of features to keep; 150-200 works well, maybe even try 250 for final submission)
selected_features = scores_df.head(K).index.tolist()

features = features_raw[selected_features]   # reduced raw features
label    = label_raw                         # keep label aligned

print("Reduced feature matrix:", features.shape)
features.head()

In [ ]:
# total number of features
total_features = features.shape[1]

# professor-provided features (start with "feature")
prof_features = [c for c in features.columns if str(c).startswith("feature_")]
n_prof = len(prof_features)

# student-engineered features (everything else)
n_engineered = total_features - n_prof

# percentages
pct_prof = 100 * n_prof / total_features
pct_engineered = 100 * n_engineered / total_features

print(f"Total features: {total_features}")
print(f"Provided features: {n_prof} ({pct_prof:.1f}%)")
print(f"Engineered features: {n_engineered} ({pct_engineered:.1f}%)")

In [ ]:
engineered_features = [c for c in features.columns if not str(c).startswith("feature")]

print(f"Number of engineered features: {len(engineered_features)}")
engineered_features

<h4>Features Pre-processing</h4>

<br> 

In [ ]:
import gc

# 1. OPTIMIZATION: Convert input to float32 immediately to save 50% RAM
# We use the existing 'features' dataframe directly
features = features.astype('float32')

print("Starting in-place normalization...")

# 2. Iterate through columns and normalize them one by one
# We overwrite the column in 'features' directly to avoid creating a second copy of the 20GB dataset
for col in features.columns:
    
    # A. Extract single column (very low memory)
    # unstack() makes it Time x Asset
    series_wide = features[col].unstack()
    
    # B. Rank (Cross-sectional)
    # pct=True gives 0.0 to 1.0
    series_wide = series_wide.rank(axis=1, pct=True)
    
    # C. Z-Score (Mean 0, Std 1)
    # Subtract mean, divide by std
    mu = series_wide.mean(axis=1)
    sigma = series_wide.std(axis=1).replace(0, 1)
    
    series_wide = series_wide.sub(mu, axis=0).div(sigma, axis=0)
    
    # D. Stack back and overwrite the original column
    # This keeps memory usage flat instead of doubling it
    features[col] = series_wide.stack().reindex(features.index).fillna(0).astype('float32')
    
    # E. Force garbage collection every few columns to keep RAM clean
    if features.columns.get_loc(col) % 10 == 0:
        gc.collect()

print("Normalization complete.")

# 3. Rename variable to match your next cell's expectation
# Since we modified 'features' in place, we just point the new name to it
features_normalized = features

# 4. Filter indices (Your original logic)
common_index = label.index.intersection(features_normalized.index)
features_normalized_train = features_normalized.reindex(common_index)
label = label.reindex(common_index)

print("Pre-processing finished. RAM is stable.")

In [ ]:
# force common index (absolute safety)
common_index = features_normalized.index.intersection(label.index)

# Create the frozen features matrix
features_frozen = (
    features_normalized
    .reindex(common_index)
    .sort_index()
)

# convert to numpy (speed)
# float32 is critical here for memory
X_all = features_frozen.values.astype(np.float32)

# store index once
index_all = features_frozen.index

# global next-hour label (built once)
# FIX: Use 'label' instead of 'data' (which was deleted)
# 'label' already contains the shifted returns we need
y_all = (
    label
    .reindex(index_all)
    .astype(np.float32)
)

# sanity checks
assert len(index_all) == X_all.shape[0]
assert len(index_all) == y_all.shape[0]

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("Number of features:", X_all.shape[1])

## Walk-forward Setup

Let $t$ denote clock time and let the full dataset be indexed by a strictly increasing time index $\{t_1, t_2, \dots, t_T\}$. As a safety condition, we explicitly verify that the time index is monotonic, ensuring that no time reordering or leakage can occur.

Rebalancing dates are defined at a **monthly frequency**, using month-end timestamps between the start of the training period and the end of the validation period. The first two month-end dates are discarded to guarantee that sufficient historical data are available for the initial training window.

For each rebalancing date $t^{(k)}$, we construct one walk-forward fold composed of a training window followed immediately by a validation window. The training window spans approximately one year of historical data and is defined as
$$
[t^{(k)} - 12\text{ months},\; t^{(k)}],
$$
while the validation window starts strictly after the training window and extends for roughly one month:
$$
(t^{(k)},\; t^{(k)} + 31\text{ days}].
$$
The one-hour gap between the end of the training window and the start of the validation window ensures that no label information from the training boundary can leak into the validation period.

In [ ]:
# Walk-forward folds

time_index = index_all.get_level_values(0) # time level
assert time_index.is_monotonic_increasing # safety check

# monthly rebalancing dates
rebalancing_dates = pd.date_range(
    start=start_date_train,
    end=last_date_validate,
    freq="ME",
)[2:]

folds = []

for last_date_train_fold in rebalancing_dates:

    # Train / validation windows
    start_date_train_fold = last_date_train_fold - pd.Timedelta(days=30 * 12)
    start_date_validate_fold = last_date_train_fold + pd.Timedelta(hours=1)
    last_date_validate_fold = last_date_train_fold + pd.Timedelta(days=31)

    # Convert dates → integer positions
    train_start = time_index.searchsorted(start_date_train_fold, side="left")
    train_end   = time_index.searchsorted(last_date_train_fold, side="right")

    valid_start = time_index.searchsorted(start_date_validate_fold, side="left")
    valid_end   = time_index.searchsorted(last_date_validate_fold, side="right")

    # Skip degenerate folds
    if train_end <= train_start or valid_end <= valid_start:
        continue

    folds.append({
        "train_start": train_start,
        "train_end": train_end,
        "valid_start": valid_start,
        "valid_end": valid_end,
    })

print(f"Number of walk-forward folds: {len(folds)}")
print("First fold:", folds[0])

## Model 1 - ElasticNet

For each walk-forward fold, we estimate a linear predictive model using an Elastic Net specification and generate out-of-sample predictions for the subsequent validation window. The model is re-estimated independently at each rebalancing date using only historical information available at that point in time, ensuring strict temporal causality.

Let $X_{t,i} \in \mathbb{R}^p$ denote the vector of selected features for asset $i$ at time $t$, and let $y_{t,i}$ denote the corresponding next-hour return. For a given fold $k$, the training sample consists of observations
$$
\{(X_{t,i},\, y_{t,i}) : t \in \mathcal{T}^{(k)}_{\text{train}}\},
$$
while predictions are generated for feature vectors in the validation window
$$
\{X_{t,i} : t \in \mathcal{T}^{(k)}_{\text{valid}}\}.
$$

The predictive model is an Elastic Net regression, defined as the solution to
$$
\hat{\beta}
=
\arg\min_{\beta}
\left\{
\frac{1}{2N}
\sum_{t,i}
\left(y_{t,i} - X_{t,i}^\top \beta\right)^2
+
\alpha
\left[
(1-\rho)\frac{1}{2}\|\beta\|_2^2
+
\rho\|\beta\|_1
\right]
\right\},
$$
where $\alpha > 0$ controls the overall regularization strength and $\rho \in [0,1]$ determines the balance between $\ell_1$ and $\ell_2$ penalties. This formulation promotes sparsity while remaining stable under strong cross-feature correlation, which is particularly important given the large and highly collinear feature set.

In [ ]:
from joblib import Parallel, delayed
from sklearn.linear_model import ElasticNet
from threadpoolctl import threadpool_limits

# ElasticNet
# ALPHA = BEST_ALPHA
# L1_RATIO = BEST_L1_RATIO

ALPHA = 3e-06
L1_RATIO = 0.45   
MAX_ITER = 500
TOL = 2e-2

def train_predict_one_month_thread(fold):
    # keep internal libs single-threaded to avoid oversubscription
    with threadpool_limits(limits=1):

        # training indices
        idx_train = index_all[fold["train_start"]:fold["train_end"]]
        idx_valid = index_all[fold["valid_start"]:fold["valid_end"]]

        # features
        X_train = X_all[fold["train_start"]:fold["train_end"]]
        X_valid = X_all[fold["valid_start"]:fold["valid_end"]]

        # label: slice from precomputed global label
        y_train = y_all[fold["train_start"]:fold["train_end"]]

        # CRITICAL: remove boundary look-ahead
        last_train_time = idx_train.get_level_values(0)[-1]
        boundary_mask = (
            idx_train.get_level_values(0) == last_train_time
        )

        y_train = y_train.copy()
        y_train[boundary_mask] = np.nan

        model = ElasticNet(
            alpha=ALPHA,
            l1_ratio=L1_RATIO,
            fit_intercept=True,
            max_iter=MAX_ITER,
            tol=TOL,
            selection="cyclic",
        )

        # drop NaNs (including boundary)
        mask = np.isfinite(y_train)

        X_train = X_train[mask]
        y_train = y_train[mask]

        model.fit(X_train, y_train)
        preds = model.predict(X_valid).astype(np.float32)

        return pd.Series(preds, index=idx_valid)

t1 = time.time()

# choose a sensible thread count
n_jobs = min(4, len(folds))
print(f"Running THREAD parallelism with n_jobs={n_jobs}")

predictions_list = Parallel(
    n_jobs=n_jobs,
    prefer="processes",
    batch_size=1
)(
    delayed(train_predict_one_month_thread)(fold) for fold in folds
)

t2 = time.time()
print(f"Elastic Net training time (threads): {t2 - t1:.2f} seconds")

predictions_df_threads = (
    pd.concat(predictions_list)
      .groupby(level=[0, 1])   # (time, asset)
      .sum()
      .unstack()
      .sort_index()
)

print(predictions_df_threads.shape)
predictions_df_threads.dropna(how="all").head()

In [ ]:
expected_returns_valid = predictions_df_threads.loc[
    start_date_validate:last_date_validate
]

returns_valid = data["return"].loc[
    start_date_validate:last_date_validate
]

analyze_expected_returns(
    expected_returns=expected_returns_valid,
    returns=returns_valid,
    rfr_hourly=rfr_hourly,
    title="Elastic Net (validation, walk-forward)",
    lags=[0,1,2,3,6,12],
    tc=tc,
)

In [ ]:
# check
expected_returns_delayed = predictions_df_threads.shift(1)

In [ ]:
analyze_expected_returns(
    expected_returns=expected_returns_delayed.loc[
        start_date_validate:last_date_validate
    ],
    returns=data["return"].loc[
        start_date_validate:last_date_validate
    ],
    rfr_hourly=rfr_hourly,
    title="Elastic Net (signal delayed by 1h)",
    lags=[0,1,2,3,6,12],
    tc=tc,
)

We introduce a one-hour delay in the trading signal. Performance decreases materially and shifts consistently across lags, while preserving the monotonic decay pattern. This behavior is consistent with short-horizon alpha decay rather than information leakage, indicating that the original results are not driven by look-ahead bias but by optimistic execution assumptions at very short horizons.

In [ ]:
positions = expected_returns_to_positions(
    predictions_df_threads.loc[start_date_validate:last_date_validate]
)

turnover_hourly = (
    positions
    .fillna(0)
    .diff()
    .abs()
    .sum(axis=1)
)

int(turnover_hourly.mean())

In [ ]:
turnover_daily = turnover_hourly.mean() * 24
int(turnover_daily)

In [ ]:
(turnover_hourly
 .rolling(24)
 .mean()
 .plot(
     title="24h Rolling Average Portfolio Turnover",
     figsize=(10,4)
 ))
plt.show()

## Model 2 - LNLM (ElasticNet linear + Hermite non-linear mix)

This model is inspired by the **LNLM philosophy** presented in *Artificial Intelligence for Financial Markets: The Polymodel* (Barrau & Douady), and follows the course’s walk-forward discipline: we first fit a strong **linear backbone** and then learn a **low-dimensional nonlinear correction** on top of the linear score. The key idea is that in many financial prediction problems, most of the stable structure is captured by a regularized linear model, while residual nonlinearities can be modeled more safely by restricting the nonlinear part to a controlled functional family and calibrating its contribution conservatively.

Let $X_{t,i} \in \mathbb{R}^p$ denote the feature vector for asset $i$ at time $t$, and let $y_{t,i}$ denote the next-hour return. For each walk-forward fold, we estimate a linear Elastic Net model on the training window, producing linear predictions
$$
\hat y^{\text{lin}}_{t,i} = X_{t,i}^\top \hat\beta,
$$
where $\hat\beta$ solves the Elastic Net objective
$$
\hat{\beta}
=
\arg\min_{\beta}
\left\{
\frac{1}{2N}\sum_{t,i}\left(y_{t,i} - X_{t,i}^\top \beta\right)^2
+
\alpha\left[(1-\rho)\frac{1}{2}\|\beta\|_2^2 + \rho\|\beta\|_1\right]
\right\}.
$$

The LNLM step then treats the **linear prediction itself** as a one-dimensional “state” variable summarizing the information in $X$. Denote the linear scores on the training set by $\hat y^{\text{lin}}$ and standardize them:
$$
z_{t,i} = \frac{\hat y^{\text{lin}}_{t,i} - m}{s},
\qquad
m = \mathbb{E}[\hat y^{\text{lin}}],\quad s = \operatorname{Std}(\hat y^{\text{lin}}),
$$
with a numerical safeguard $s \leftarrow 1$ if $s$ is zero or non-finite. The nonlinear correction is modeled as a truncated expansion in **probabilists’ Hermite polynomials** $\{\mathrm{He}_u(\cdot)\}_{u=1}^U$, with $U=4$ (a low degree recommended as sufficient in practice in the LNLM/Polymodel methodology). The Hermite basis is defined recursively by
$$
\mathrm{He}_0(x)=1,\qquad \mathrm{He}_1(x)=x,\qquad \mathrm{He}_u(x)=x\,\mathrm{He}_{u-1}(x)-(u-1)\,\mathrm{He}_{u-2}(x),
$$
and we construct the design matrix
$$
H(z) = \big[\mathrm{He}_1(z),\ldots,\mathrm{He}_U(z)\big]\in\mathbb{R}^{N\times U},
$$
without an intercept. The nonlinear model is trained on **linear residuals**
$$
\varepsilon_{t,i} = y_{t,i} - \hat y^{\text{lin}}_{t,i},
$$
by fitting a small OLS regression (via least squares) on the Hermite features:
$$
\hat\theta = \arg\min_{\theta\in\mathbb{R}^U} \sum_{t,i}\left(\varepsilon_{t,i} - H(z_{t,i})^\top \theta\right)^2,
\qquad
\widehat{\varepsilon}^{\text{nl}}_{t,i}=H(z_{t,i})^\top \hat\theta.
$$
This yields a nonlinear correction term $\widehat{\varepsilon}^{\text{nl}}$ that depends only on the one-dimensional score $z$, which is a deliberate complexity control: the nonlinear part cannot freely overfit the full $p$-dimensional space, it can only learn a smooth nonlinear mapping of the linear predictor.

The final LNLM forecast combines the linear and nonlinear components through a scalar mixing coefficient $\mu$:
$$
\hat y^{\text{LNLM}}_{t,i} = \hat y^{\text{lin}}_{t,i} + \mu \,\widehat{\varepsilon}^{\text{nl}}_{t,i}.
$$
Rather than fixing $\mu$ arbitrarily, we calibrate it *inside each walk-forward fold* using a small **internal stratified $K$-fold procedure** (here $K=5$) applied to the training observations. Stratification is done by sorting the training targets $y$ and assigning fold IDs in small quantile buckets, which keeps each internal fold approximately balanced across the return distribution. For each internal split, we fit the nonlinear residual regression on the internal sub-train sample, predict residuals on the internal sub-validation sample, and then search over a grid $\mu\in[-2,2]$ to pick the value that optimizes a ranking-relevant criterion. Concretely, we maximize the Spearman information coefficient between the combined prediction and the realized returns on the internal validation sample:
$$
\mu^\star = \arg\max_{\mu\in\mathcal{G}} \ \rho_S\!\left(\hat y^{\text{lin}} + \mu\,\widehat{\varepsilon}^{\text{nl}},\ y\right),
$$
where $\rho_S(\cdot,\cdot)$ is Spearman correlation and $\mathcal{G}$ is the MU_GRID. Alongside $\mu^\star$, we compute a dispersion proxy $\xi$ measuring how sharply peaked the objective is around its optimum (here implemented as the standard deviation of objective values relative to the best), and then aggregate the fold-specific $\mu^\star$ values into a final $\mu$ using inverse-dispersion weighting:
$$
\mu_{\text{final}} = \frac{\sum_{k} w_k \mu_k}{\sum_k w_k},
\qquad
w_k = \frac{1}{\xi_k + \epsilon}.
$$
This weighting emphasizes internal splits where the optimum is more stable (smaller dispersion), which is consistent with the conservative calibration spirit emphasized in the Polymodel/LNLM approach.

Once $\mu_{\text{final}}$ is chosen for the walk-forward fold, we refit the nonlinear residual regression on the full training residuals, generate nonlinear residual forecasts on the fold’s validation window using the standardized linear scores $z_{\text{valid}}$, and output the final validation predictions
$$
\hat y^{\text{LNLM}}_{\text{valid}} = \hat y^{\text{lin}}_{\text{valid}} + \mu_{\text{final}}\,\widehat{\varepsilon}^{\text{nl}}_{\text{valid}}.
$$

Conceptually, this LNLM construction implements the Polymodel/LNLM intuition in a controlled way: use a regularized linear model to extract a stable signal, then let a low-degree orthogonal polynomial expansion capture smooth nonlinear distortions of that signal, and calibrate the nonlinear contribution via an internal procedure aligned with the portfolio objective (rank-based association with future returns), all while maintaining strict walk-forward separation as emphasized in the course methodology.

In [ ]:
# LNLM hyperparams (fast + robust defaults) (takes 30-40 min to fit)

U = 4                      # Hermite degree (book says 4 usually enough)
K_FOLDS = 5                # stratified folds used to pick mu inside each WF fold
MU_GRID = np.linspace(-2.0, 2.0, 161)  # closer to book (finer search)
SEED = 0

# Hermite polynomials (probabilists' Hermite He_n)
# He_0=1, He_1=x, He_{n}=x*He_{n-1}-(n-1)*He_{n-2}

def hermite_design(x: np.ndarray, degree: int) -> np.ndarray:
    """
    return design matrix [He_1(x), ..., He_degree(x)] with no intercept
    x: (n,)
    """
    x = x.astype(np.float64, copy=False)
    n = x.shape[0]
    H = np.empty((n, degree), dtype=np.float64)

    if degree >= 1:
        H[:, 0] = x  # He_1
    if degree >= 2:
        He_nm2 = np.ones(n, dtype=np.float64)  # He_0
        He_nm1 = x.copy()                      # He_1
        for k in range(2, degree + 1):
            He_n = x * He_nm1 - (k - 1) * He_nm2
            H[:, k - 1] = He_n
            He_nm2, He_nm1 = He_nm1, He_n
    return H

def ols_fit_predict(Xtr: np.ndarray, ytr: np.ndarray, Xte: np.ndarray) -> np.ndarray:
    """
    OLS via lstsq (tiny matrices here => very fast)
    """
    beta, *_ = np.linalg.lstsq(Xtr, ytr, rcond=None)
    return Xte @ beta

# Stratified K-fold assignment (quantile bucket method)

def stratified_folds_indices(y: np.ndarray, k: int, seed: int = 0) -> np.ndarray:
    """
    return fold_id per observation in {0,...,k-1} using quantile buckets of size k
    """
    rng = np.random.default_rng(seed)
    n = y.shape[0]
    order = np.argsort(y)
    fold_id = np.empty(n, dtype=np.int32)

    # buckets of size k in sorted order
    for start in range(0, n, k):
        bucket = order[start:start + k]
        # assign unique fold ids inside bucket (as much as possible)
        ids = np.arange(k, dtype=np.int32)
        rng.shuffle(ids)
        fold_id[bucket] = ids[:bucket.shape[0]]
    return fold_id

def choose_mu_lnlm_residual(
    y_true: np.ndarray,
    y_lin: np.ndarray,
    res_nonlin: np.ndarray,
) -> tuple[float, float]:
    """
    residual LNLM: y_hat = y_lin + mu * res_nonlin
    choose mu minimizing RMSE on the validation split
    """
    rmse_vals = []
    for mu in MU_GRID:
        y_hat = y_lin + mu * res_nonlin
        rmse_vals.append(np.sqrt(np.mean((y_true - y_hat) ** 2)))

    rmse_vals = np.asarray(rmse_vals, dtype=np.float64)
    j = int(np.argmin(rmse_vals))
    mu_star = float(MU_GRID[j])
    r_star = float(rmse_vals[j])

    xi = float(np.std(rmse_vals - r_star))
    return mu_star, xi

def choose_mu_max_spearman_ic(y_true, y_lin, res_nonlin):
    best_mu = 0.0
    best_ic = -1e9
    ic_vals = []

    for mu in MU_GRID:
        y_hat = y_lin + mu * res_nonlin
        ic = spearmanr(y_hat, y_true).correlation
        if np.isnan(ic):
            ic = -1e9
        ic_vals.append(ic)
        if ic > best_ic:
            best_ic = ic
            best_mu = float(mu)

    ic_vals = np.asarray(ic_vals, dtype=np.float64)
    # “xi” analogue: dispersion around best
    xi = float(np.std(ic_vals - best_ic))
    return best_mu, xi


def train_predict_one_month_lnlm_thread(fold):
    with threadpool_limits(limits=1):
        
        # indices (needed for boundary purge)
        idx_train = index_all[fold["train_start"]:fold["train_end"]]
        idx_valid = index_all[fold["valid_start"]:fold["valid_end"]]

        # slice train/valid
        X_train = X_all[fold["train_start"]:fold["train_end"]]
        y_train = y_all[fold["train_start"]:fold["train_end"]]
        X_valid = X_all[fold["valid_start"]:fold["valid_end"]]

        # remove boundary look-ahead
        last_train_time = idx_train.get_level_values(0)[-1]
        boundary_mask = (idx_train.get_level_values(0) == last_train_time)

        y_train = y_train.copy()
        y_train[boundary_mask] = np.nan

        mask = np.isfinite(y_train)
        X_train = X_train[mask]
        y_train = y_train[mask]

        # linear model (ElasticNet)
        lin = ElasticNet(
            alpha=ALPHA,
            l1_ratio=L1_RATIO,
            fit_intercept=True,
            max_iter=MAX_ITER,
            tol=TOL,
            selection="cyclic",
        )
        
        lin.fit(X_train, y_train)
        y_lin_train = lin.predict(X_train).astype(np.float64)
        y_lin_valid = lin.predict(X_valid).astype(np.float64)

        # build non-linear regressor input = standardized linear score
        m = y_lin_train.mean()
        s = y_lin_train.std()
        if s == 0 or not np.isfinite(s):
            s = 1.0

        z_train = (y_lin_train - m) / s
        z_valid = (y_lin_valid - m) / s

        H_train = hermite_design(z_train, degree=U)  # (n_train, U)
        H_valid = hermite_design(z_valid, degree=U)  # (n_valid, U)

        # choose mu via stratified k-fold inside the TRAIN window
        fold_ids = stratified_folds_indices(y_train.astype(np.float64), k=K_FOLDS, seed=SEED)

        mu_list = []
        xi_list = []

        for k in range(K_FOLDS):
            val_mask = (fold_ids == k)
            tr_mask = ~val_mask

            if tr_mask.sum() < U + 5 or val_mask.sum() < 50:
                continue

            # fit NonLin OLS on train-subsample (predict on val-subsample)
            res_tr = (y_train[tr_mask].astype(np.float64) - y_lin_train[tr_mask])
            y_nonlin_val = ols_fit_predict(
                Xtr=H_train[tr_mask],
                ytr=res_tr,             # residuals, no intercept
                Xte=H_train[val_mask],
            )
            y_lin_val = y_lin_train[val_mask]
            y_true_val = y_train[val_mask].astype(np.float64)

            mu_k, xi_k = choose_mu_max_spearman_ic(
                y_true=y_true_val,
                y_lin=y_lin_val,
                res_nonlin=y_nonlin_val,
            )
            mu_list.append(mu_k)
            xi_list.append(xi_k)

        if len(mu_list) == 0:
            mu_final = 0.0  # fallback: pure linear
        else:
            mu_arr = np.asarray(mu_list, dtype=np.float64)
            xi_arr = np.asarray(xi_list, dtype=np.float64)
            if np.all(xi_arr == 0):
                mu_final = float(mu_arr.mean())
            else:
                w = 1.0 / (xi_arr + 1e-12)
                mu_final = float(np.sum(mu_arr * w) / np.sum(w))


        # refit NonLin on full train residuals; predict valid
        res_full = (y_train.astype(np.float64) - y_lin_train)   # residuals vs linear fit

        res_nonlin_valid = ols_fit_predict(
                Xtr=H_train,
                ytr=res_full,    # residual target
                Xte=H_valid,
        )

        y_pred_valid = y_lin_valid + mu_final * res_nonlin_valid
        y_pred_valid = y_pred_valid.astype(np.float32)

        return pd.Series(y_pred_valid, index=idx_valid), mu_final


# Run WF folds in parallel

t1 = time.time()
n_jobs = min(len(folds), os.cpu_count() or 1)
print(f"Running LNLM (ElasticNet + Hermite) with n_jobs={n_jobs}")

pred_list = Parallel(n_jobs=n_jobs, prefer="threads", batch_size=1)(
    delayed(train_predict_one_month_lnlm_thread)(fold) for fold in folds
)

t2 = time.time()
print(f"LNLM training time (threads): {t2 - t1:.2f} seconds")

# pred_list is a list of (Series, mu)
pred_series_list, mu_per_fold = zip(*pred_list)

mu_per_fold = np.array(mu_per_fold, dtype=float)
print("mu summary:",
      "mean=", mu_per_fold.mean(),
      "std=", mu_per_fold.std(),
      "min=", mu_per_fold.min(),
      "max=", mu_per_fold.max())

predictions_df_lnlm = (
    pd.concat(pred_series_list)
      .groupby(level=[0, 1])
      .sum()
      .unstack()
      .sort_index()
)

print("predictions_df_lnlm shape:", predictions_df_lnlm.shape)
predictions_df_lnlm.dropna(how="all").head()

In [ ]:
print("unique mu (rounded):", np.unique(np.round(mu_per_fold, 3))[:30])

In [ ]:
analyze_expected_returns(
    expected_returns=predictions_df_lnlm.loc[start_date_validate:last_date_validate],
    returns=data["return"].loc[start_date_validate:last_date_validate],
    rfr_hourly=rfr_hourly,
    title="LNLM (ElasticNet + Hermite NonLin mix)",
    lags=[0,1,2,3,6,12],
    tc=tc,
)

## Model 3a - XGBoost

In [ ]:
import time
import pandas as pd
import numpy as np
import os
import xgboost
from xgboost import XGBRegressor
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURATION ---
data_path = "/Users/veli/Desktop/Misc/Python Scritps /QuantTradingStrategies/in_sample/"

# Your Stratified Selection (15 Conditioners + 35 Features)
top_50_features = [
    'conditioner_273515112431', 'conditioner_383133072066', 'conditioner_914020979387', 
    'conditioner_833295758790', 'conditioner_959064417012', 'conditioner_894036834441', 
    'conditioner_686927401640', 'conditioner_837842304804', 'conditioner_781989077658', 
    'conditioner_540918037375', 'conditioner_729111636542', 'conditioner_335548928575', 
    'conditioner_490956708078', 'conditioner_137846269002', 'conditioner_602562914883', 
    'feature_811799367203', 'feature_847894150363', 'feature_962320961442', 
    'feature_687591005636', 'feature_447214984470', 'feature_111017065985', 
    'feature_254273731453', 'feature_397445941179', 'feature_955845031686', 
    'feature_858306317837', 'feature_361022551287', 'feature_422901494305', 
    'feature_345829329118', 'feature_632247244915', 'feature_163039513711', 
    'feature_004212234751', 'feature_583653370974', 'feature_577482951771', 
    'feature_907679094852', 'feature_576016642244', 'feature_889641854357', 
    'feature_766607522404', 'feature_814297418233', 'feature_351107626410', 
    'feature_002321956187', 'feature_881677815989', 'feature_657157666184', 
    'feature_458516196156', 'feature_300199210163', 'feature_909460334104', 
    'feature_373101088533', 'feature_840341828987', 'feature_566747201659', 
    'feature_711197486128', 'feature_859096531804'
]

# --- 1. LOAD & PREPARE DATA ---
print(f"Loading {len(top_50_features)} selected inputs (Features + Conditioners)...")
feature_dict = {}

# We only need ONE loop now because your list contains both filenames.
for feat_name in top_50_features:
    try:
        # Load the file matching the list name
        path = os.path.join(data_path, f"{feat_name}.csv")
        df = pd.read_csv(path, index_col=0, header=[0])
        df.index = pd.to_datetime(df.index)
        feature_dict[feat_name] = df.stack()
    except Exception as e:
        print(f"Warning: Could not load {feat_name}: {e}")

# Create DataFrame
features_df = pd.DataFrame(feature_dict)

# Normalize (Rank -> Center)
# Since features and conditioners are now in one dataframe, they get normalized together.
features_normalized = features_df.unstack().rank(axis=1, pct=True) - 0.5
features_normalized = features_normalized.stack().fillna(0)


# --- 2. THE COMBINED FUNCTION ---
def train_predict_period(rebalance_date, returns, hyperparameters):
    """
    Rolling Window: Train on [T-12 Months, T] -> Predict [T, T+1 Month]
    With strict 'Purging' to prevent look-ahead bias.
    """
    
    # A. Define Dates
    # Train Window: Past 12 Months
    start_train = rebalance_date - pd.Timedelta(days=365)
    end_train   = rebalance_date
    
    # Predict Window: Next 1 Month
    # We start prediction 1 hour AFTER the rebalance date to be safe
    start_predict = rebalance_date + pd.Timedelta(hours=1)
    end_predict   = rebalance_date + pd.Timedelta(days=31)

    print(f"Train: {start_train} -> {end_train} | Predict: {start_predict} -> {end_predict}")

    # B. Create Targets
    # shift(-1) aligns T with Return(T, T+1)
    y_train = returns.loc[start_train:end_train].shift(-1).stack()
    
    # --- C. THE PURGE (Crucial Quant Step) ---
    y_train = y_train.loc[y_train.index.get_level_values(0) < end_train]
    
    # D. Align Features
    X_train = features_normalized.reindex(y_train.index)
    
    # E. Clean Data
    # 1. Replace Inf with NaN
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    y_train = y_train.replace([np.inf, -np.inf], np.nan)
    
    # 2. Drop NaNs
    common_idx = X_train.dropna().index.intersection(y_train.dropna().index)
    X_train = X_train.loc[common_idx]
    y_train = y_train.loc[common_idx]

    # F. Prepare Prediction Set
    X_predict = features_normalized.sort_index().loc[start_predict:end_predict]
    
    # Safety Checks
    if len(X_train) < 100 or len(X_predict) == 0:
        print(f"  -> Skipping: Insufficient data (Train: {len(X_train)}, Pred: {len(X_predict)})")
        return pd.DataFrame()

    # G. Train Model
    params = hyperparameters.copy()
    params['n_jobs'] = -1 
    
    model = XGBRegressor(**params)
    
    try:
        model.fit(X_train, y_train)
        preds = model.predict(X_predict)
    except Exception as e:
        print(f"  -> Error training: {e}")
        return pd.DataFrame()
    
    # Format Result
    res = pd.Series(preds, index=X_predict.index).unstack()
    return pd.concat({str(rebalance_date): res}, axis=1)

# --- 3. EXECUTION LOOP ---
t1 = time.time()

# Define Dates: Skip first 12 months to allow for the first training window
rebalancing_dates = pd.date_range(
    start=start_date_train, 
    end=last_date_validate, 
    freq="ME"
)
# Ensure we have enough history for the first lookback
rebalancing_dates = rebalancing_dates[rebalancing_dates > (pd.to_datetime(start_date_train) + pd.Timedelta(days=360))]

print(f"Running Walk-Forward on {len(rebalancing_dates)} periods...")

#Best Params: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 
# 0.0010377686699765967, 'subsample': 0.6341753157689765, 'colsample_bytree': 0.8366208336383808, 'min_child_weight': 10}
hyperparameters = {
    "n_estimators": 500,        
    "max_depth": 10,             
    "learning_rate": 0.001,
    "subsample": 0.6342,
    "colsample_bytree": 0.8367,
    "objective": "reg:squarederror",
    "tree_method": "hist",      
    "random_state": 0,
    "min_child_weight": 10
}

predictions_list = []

for date in rebalancing_dates:
    res = train_predict_period(
        rebalance_date=date,
        returns=data["return"],
        hyperparameters=hyperparameters
    )
    if not res.empty:
        predictions_list.append(res)

# --- 4. AGGREGATE RESULTS ---
if predictions_list:
    # Combine all monthly predictions
    predictions = pd.concat(predictions_list, axis=1)
    
    # Clean up formatting (MultiIndex handling)
    predictions = predictions.T.groupby(level=1).sum().T
    predictions = predictions.replace(0, np.nan).dropna(how='all')
    
    print(f"Total Training time: {time.time()-t1:.2f} seconds")
    
    # --- 5. ANALYZE ---
    analyze_expected_returns(
        expected_returns=predictions.loc[start_date_validate:last_date_validate],
        returns=data["return"].loc[start_date_validate:last_date_validate],
        rfr_hourly=rfr_hourly,
        title=f"XGBoost Stratified (Conditioners + Features)",
        lags=[0,1,2,3,6,12],
        tc=0.0,
    )
else:
    print("No predictions generated. Check date ranges.")

## Model 3 - ElasticNet trend + XGBoost residuals

This model implements a **hybrid forecasting architecture** motivated by a well-known limitation of tree-based models in time-series settings: gradient-boosted trees cannot extrapolate beyond the range of values observed during training. Because predictions are averages of leaf values, tree-based models are inherently bounded by the training target distribution. In the presence of trends or non-stationarity, this leads to flat or lagging forecasts when test values exceed the historical range.

The linear model captures **persistent, approximately linear structure** in the data, including slow-moving trends and cross-sectional relationships that benefit from extrapolation. We then compute training residuals:
$$
\varepsilon_{t,i} = y_{t,i} - \hat y^{\text{lin}}_{t,i}.
$$
These residuals are, by construction, more stationary and centered than the raw target, making them more suitable for tree-based learning.

In the second stage, we train a gradient-boosted decision tree model (XGBoost) on the same feature set $X_{t,i}$, but with the residuals $\varepsilon_{t,i}$ as the target:
$$
\hat \varepsilon^{\text{gb}}_{t,i} = f_{\text{GB}}(X_{t,i}),
$$
where $f_{\text{GB}}$ denotes an ensemble of shallow regression trees trained with squared-error loss and conservative regularization. The hyperparameters are chosen to favor robustness over expressiveness, with limited depth, subsampling, and minimum child weight to reduce overfitting.

The final hybrid prediction on the validation window is obtained by recombining the two components:
$$
\hat y^{\text{hybrid}}_{t,i}
=
\hat y^{\text{lin}}_{t,i}
+
\hat \varepsilon^{\text{gb}}_{t,i}.
$$
This ensures that long-horizon extrapolation is handled by the linear component, while the boosting model contributes only bounded nonlinear corrections around the trend. Importantly, the tree-based model is never asked to extrapolate the level of returns; it only learns patterns in deviations from the linear baseline.

Conceptually, this model is complementary to the LNLM framework used previously. While LNLM restricts nonlinearities to a low-dimensional, orthogonal polynomial expansion of the linear score, the hybrid model allows for richer nonlinear interactions through boosted trees, at the cost of weaker interpretability. Both approaches share the same core philosophy: preserve the extrapolation properties of linear models and delegate nonlinear structure to a secondary correction layer.

In [ ]:
# XGBoost params (for residuals)
XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    gamma=0.1,
    max_leaves=64,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    n_jobs=1,
    random_state=0,
)

def train_predict_one_month_hybrid(fold):

    with threadpool_limits(limits=1):

        # indices (needed for boundary purge)
        idx_train = index_all[fold["train_start"]:fold["train_end"]]
        idx_valid = index_all[fold["valid_start"]:fold["valid_end"]]

        # split data
        X_train = X_all[fold["train_start"]:fold["train_end"]]
        y_train = y_all[fold["train_start"]:fold["train_end"]]

        X_valid = X_all[fold["valid_start"]:fold["valid_end"]]

        # remove boundary look-ahead
        last_train_time = idx_train.get_level_values(0)[-1]
        boundary_mask = (idx_train.get_level_values(0) == last_train_time)

        y_train = y_train.copy()
        y_train[boundary_mask] = np.nan

        mask = np.isfinite(y_train)
        X_train = X_train[mask]
        y_train = y_train[mask]

        # linear model (trend)
        enet = ElasticNet(
            alpha=ALPHA,
            l1_ratio=L1_RATIO,
            fit_intercept=True,
            max_iter=MAX_ITER,
            tol=TOL,
            selection="cyclic",
        )

        enet.fit(X_train, y_train)

        y_train_linear = enet.predict(X_train)
        y_valid_linear = enet.predict(X_valid)

        # residuals
        residuals_train = y_train - y_train_linear

        # XGBoost on residuals
        xgb = XGBRegressor(**XGB_PARAMS)
        xgb.fit(X_train, residuals_train, verbose=False)

        residuals_valid_pred = xgb.predict(X_valid)

        # final prediction
        y_valid_pred = y_valid_linear + residuals_valid_pred

        return pd.Series(y_valid_pred.astype(np.float32), index=idx_valid)

# Run in parallel
t1 = time.time()

n_jobs = min(len(folds), os.cpu_count() or 1)
print(f"Running HYBRID model with n_jobs={n_jobs}")

predictions_list = Parallel(
    n_jobs=n_jobs,
    prefer="threads",
    batch_size=1
)(
    delayed(train_predict_one_month_hybrid)(fold)
    for fold in folds
)

t2 = time.time()
print(f"Hybrid model training time: {t2 - t1:.2f} seconds")

# Aggregate predictions
predictions_df_hybrid = (
    pd.concat(predictions_list)
    .groupby(level=[0, 1])
    .sum()
    .unstack()
    .sort_index()
)

print(predictions_df_hybrid.shape)

In [ ]:
analyze_expected_returns(
    expected_returns=predictions_df_hybrid.loc[start_date_validate:last_date_validate],
    returns=data["return"].loc[start_date_validate:last_date_validate],
    rfr_hourly=rfr_hourly,
    title="Hybrid ElasticNet + XGBoost",
    lags=[0,1,2,3,6,12],
    tc=tc,
)

## Model 4 - LSTM (Long Short-Term Memory)

We introduce a deep learning approach using an LSTM network designed for financial time-series forecasting. While the linear and hybrid models capture cross-sectional and instantaneous relationships efficiently, they do not explicitly model the temporal evolution of features for a specific asset over time.

An LSTM is capable of learning order dependence in sequence prediction problems. It works by maintaining an internal state (cell state) that "remembers" relevant context (like momentum or volatility regimes) over the sequence window.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import time
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# --- Hyperparameters ---
LSTM_SEQ_LEN = 24
LSTM_BATCH_SIZE = 256
LSTM_EPOCHS = 7
LSTM_LR = 1e-3
LSTM_HIDDEN_DIM = 32
LSTM_LAYERS = 1

# --- Model Definition ---
class LSTMRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1, dropout=0.0):
        super(LSTMRegressor, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :]) 
        return out

# --- Data Helper Functions ---
def build_parent_pointers(index_s):
    df_map = pd.DataFrame({'idx': np.arange(len(index_s))}, index=index_s)
    prev_indices = df_map.groupby(level=1)['idx'].shift(1).fillna(-1).astype(int).values
    return prev_indices

class FinancialSequenceDataset(Dataset):
    def __init__(self, X_data, y_data, indices, parent_pointers, seq_len):
        self.X_data = X_data
        self.y_data = y_data if y_data is None else np.array(y_data)
        self.indices = indices
        self.parent_pointers = parent_pointers
        self.seq_len = seq_len

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        target_row_idx = self.indices[idx]
        seq = np.zeros(self.seq_len, dtype=int)
        seq[-1] = target_row_idx
        
        curr = target_row_idx
        for i in range(self.seq_len - 2, -1, -1):
            prev = self.parent_pointers[curr]
            if prev == -1:
                seq[i] = curr
            else:
                seq[i] = prev
                curr = prev
        
        x_seq = self.X_data[seq]
        
        if self.y_data is not None:
            y_val = self.y_data[target_row_idx]
            return torch.tensor(x_seq, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
        else:
            return torch.tensor(x_seq, dtype=torch.float32)

# --- Training Loop ---
def train_predict_one_month_lstm(fold, parent_pointers):
    # CRITICAL FIX: Limit PyTorch to 1 thread per job to prevent CPU oversubscription
    torch.set_num_threads(1) 
    
    device = torch.device("cpu")
    
    idx_train = index_all[fold["train_start"]:fold["train_end"]]
    last_train_time = idx_train.get_level_values(0)[-1]
    boundary_mask = (idx_train.get_level_values(0) == last_train_time)
    
    y_train_check = y_all[fold["train_start"]:fold["train_end"]]
    valid_train_mask = np.isfinite(y_train_check) & (~boundary_mask)
    
    train_indices_full = np.arange(fold["train_start"], fold["train_end"])
    train_indices_filtered = train_indices_full[valid_train_mask]
    
    valid_indices = np.arange(fold["valid_start"], fold["valid_end"])
    y_all_np = y_all.values if hasattr(y_all, 'values') else y_all

    train_ds = FinancialSequenceDataset(X_all, y_all_np, train_indices_filtered, parent_pointers, LSTM_SEQ_LEN)
    valid_ds = FinancialSequenceDataset(X_all, y_all_np, valid_indices, parent_pointers, LSTM_SEQ_LEN)
    
    train_loader = DataLoader(train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=False, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=LSTM_BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = LSTMRegressor(
        input_dim=X_all.shape[1], 
        hidden_dim=LSTM_HIDDEN_DIM, 
        num_layers=LSTM_LAYERS
    ).to(device)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_LR)

    model.train()
    for _ in range(LSTM_EPOCHS):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch).squeeze()
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            
    model.eval()
    preds_collect = []
    with torch.no_grad():
        for X_batch, _ in valid_loader:
            X_batch = X_batch.to(device)
            out = model(X_batch).squeeze()
            if out.ndim == 0: out = out.unsqueeze(0)
            preds_collect.append(out.cpu().numpy())
            
    if len(preds_collect) > 0:
        flat_preds = np.concatenate(preds_collect)
    else:
        flat_preds = np.array([])
        
    idx_valid = index_all[fold["valid_start"]:fold["valid_end"]]
    return pd.Series(flat_preds, index=idx_valid)

# --- Execution ---
if 'index_all' in globals() and 'folds' in globals():
    print("Building sequence map for LSTM...")
    parent_pointers_map = build_parent_pointers(index_all)

    print(f"Running LSTM model training with {len(folds)} folds...")
    t1_lstm = time.time()

    # Progress bar RESTORED as requested
    # Note: With torch.set_num_threads(1), correct execution is ensured.
    predictions_list_lstm = Parallel(n_jobs=-1, prefer="threads")(
        delayed(train_predict_one_month_lstm)(fold, parent_pointers_map) 
        for fold in tqdm(folds, desc="LSTM Folds", total=len(folds))
    )

    t2_lstm = time.time()
    print(f"LSTM training time: {t2_lstm - t1_lstm:.2f} seconds")

    predictions_df_lstm = (
        pd.concat(predictions_list_lstm)
        .groupby(level=[0, 1])
        .sum()
        .unstack()
        .sort_index()
    )
    print("predictions_df_lstm shape:", predictions_df_lstm.shape)
    display(predictions_df_lstm.dropna(how="all").head())
else:
    print("Skipping LSTM training: 'index_all' or 'folds' not defined. Run data block first.")
    predictions_df_lstm = pd.DataFrame()

In [ ]:
# Update the models dictionary to include LSTM
# This ensures it is included in any subsequent comparison plots/tables
if 'models' not in globals():
    # If models dict wasn't defined yet, attempt to reconstruct it from typical variable names
    # (Handling case where previous cells were not run in a way that preserved 'models' dict)
    models = {}
    if 'predictions_df_threads' in globals(): models["ElasticNet"] = predictions_df_threads
    if 'predictions_df_lnlm' in globals(): models["LNLM"] = predictions_df_lnlm
    if 'predictions_df_hybrid' in globals(): models["Hybrid"] = predictions_df_hybrid

models["LSTM"] = predictions_df_lstm

# Run the standard analysis suite for this model
analyze_expected_returns(
    expected_returns=predictions_df_lstm.loc[start_date_validate:last_date_validate],
    returns=data["return"].loc[start_date_validate:last_date_validate],
    rfr_hourly=rfr_hourly,
    title="Model 4 - LSTM",
    lags=[0,1,2,3,6,12],
    tc=tc,
)

## Comparison

In [ ]:
# Collect expected returns from all models

models = {
    "ElasticNet": predictions_df_threads,
    "LNLM": predictions_df_lnlm,
    "Hybrid_EN_XGB": predictions_df_hybrid,
    "LSTM": predictions_df_lstm,
}

returns_val = data["return"].loc[start_date_validate:last_date_validate]

In [ ]:
from support_functions import get_sharpe

def compute_finance_stats(expected_returns, returns, rfr):
    positions = expected_returns_to_positions(expected_returns)

    pnl = (
        positions.shift(1)
        .mul(returns)
        .sum(axis=1)
    )

    sharpe = get_sharpe(pnl, rfr)
    ann_ret = pnl.mean() * 24 * 365
    ann_vol = pnl.std() * np.sqrt(24 * 365)

    cum_pnl = pnl.cumsum()
    dd = cum_pnl - cum_pnl.cummax()
    max_dd = dd.min()

    turnover = (
        positions.fillna(0)
        .diff()
        .abs()
        .sum(axis=1)
        .mean() * 24
    )

    return {
        "Sharpe": sharpe,
        "Ann.Return": ann_ret,
        "Ann.Vol": ann_vol,
        "Max.Drawdown": max_dd,
        "Turnover (daily)": turnover,
        "Return / Turnover": ann_ret / turnover if turnover > 0 else np.nan,
    }

stats = []
for name, preds in models.items():
    stats.append({
        "Model": name,
        **compute_finance_stats(
            preds.loc[start_date_validate:last_date_validate],
            returns_val,
            rfr_hourly,
        )
    })

stats_df = pd.DataFrame(stats).set_index("Model")
stats_df

In [ ]:
plt.figure(figsize=(10,5))

for name, preds in models.items():
    positions = expected_returns_to_positions(
        preds.loc[start_date_validate:last_date_validate]
    )
    pnl = (
        positions.shift(1)
        .mul(returns_val)
        .sum(axis=1)
    )
    pnl.cumsum().plot(label=name)

plt.title("Cumulative PnL — Validation Period")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def rolling_sharpe(pnl, rfr, window=24*30):
    excess = pnl - rfr
    return (
        excess.rolling(window).mean()
        / excess.rolling(window).std()
        * np.sqrt(24*365)
    )

plt.figure(figsize=(10,5))

for name, preds in models.items():
    positions = expected_returns_to_positions(
        preds.loc[start_date_validate:last_date_validate]
    )
    pnl = (
        positions.shift(1)
        .mul(returns_val)
        .sum(axis=1)
    )
    rolling_sharpe(pnl, rfr_hourly).plot(label=name)

plt.title("Rolling Sharpe (30-day window)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
turnover_stats = {}

for name, preds in models.items():
    positions = expected_returns_to_positions(
        preds.loc[start_date_validate:last_date_validate]
    )
    turnover = (
        positions.fillna(0)
        .diff()
        .abs()
        .sum(axis=1)
    )
    turnover_stats[name] = turnover.mean() * 24

pd.Series(turnover_stats, name="Avg Daily Turnover")

In [ ]:
signal_corr = pd.DataFrame()

for name_i, preds_i in models.items():
    for name_j, preds_j in models.items():
        s_i = preds_i.loc[start_date_validate:last_date_validate].stack()
        s_j = preds_j.loc[start_date_validate:last_date_validate].stack()
        common = s_i.index.intersection(s_j.index)
        corr = np.corrcoef(s_i.loc[common], s_j.loc[common])[0,1]
        signal_corr.loc[name_i, name_j] = corr

signal_corr

In [ ]:
for name, preds in models.items():
    analyze_expected_returns(
        expected_returns=preds.loc[start_date_validate:last_date_validate],
        returns=returns_val,
        rfr_hourly=rfr_hourly,
        title=f"{name}",
        lags=[0,1,2,3,6,12],
        tc=tc,
    )

## Model 5 — Stacked Ensemble

Stacking (also called *stacked generalization*) is a technique that trains a **meta-learner** to optimally combine the predictions of multiple base models. Instead of simply averaging signals, a second-stage model learns *how much to trust each base model* — and whether certain models are more informative in certain regimes.

### Why stacking works here

Our four base models capture different aspects of the data:

| Base model | Strength |
|---|---|
| **ElasticNet** | Stable linear cross-sectional signal |
| **LNLM** | Smooth nonlinear correction of the linear score |
| **Hybrid (EN + XGB)** | Nonlinear feature interactions via boosted trees |
| **LSTM** | Temporal dependencies across sequential observations |

A simple average treats every model equally at every point in time. A stacked ensemble can learn, for example, that the LSTM is more useful when recent volatility is high, or that the linear model dominates in calm regimes.

### Walk-forward stacking procedure (no look-ahead)

The meta-learner must be trained on **out-of-sample predictions** from the base models to avoid information leakage. We achieve this with the following procedure:

1. **Collect base-model OOS predictions.** Each base model already produces walk-forward out-of-sample predictions on its validation window. We stack these into a meta-feature matrix $\mathbf{Z}$, where each column is one model's OOS prediction.

2. **Walk-forward meta-training.** We split the validation period into expanding meta-train / meta-validation windows (monthly rebalancing). At each step, the meta-learner (a Ridge regression) is fitted only on past OOS predictions and realized returns:
$$
\hat{w} = \arg\min_{w} \left\{ \frac{1}{N}\sum_{t,i} \left(y_{t,i} - \mathbf{z}_{t,i}^\top w\right)^2 + \lambda \|w\|_2^2 \right\}
$$

3. **Predict forward.** The fitted meta-weights are applied to the next month's base-model predictions to produce the ensemble forecast.

This ensures the meta-learner never sees future data — it only learns from base-model predictions that were themselves generated out-of-sample.

In [ ]:
from sklearn.linear_model import Ridge

# ──────────────────────────────────────────────────────────────
# 1. Build the meta-feature matrix from base-model OOS predictions
# ──────────────────────────────────────────────────────────────

base_models = {
    "ElasticNet":    predictions_df_threads,
    "LNLM":         predictions_df_lnlm,
    "Hybrid_EN_XGB": predictions_df_hybrid,
    "LSTM":         predictions_df_lstm,
}

# Stack all base-model predictions into a single DataFrame
# Each model's predictions are already OOS (walk-forward validation windows)
meta_frames = {}
for name, preds_wide in base_models.items():
    meta_frames[name] = preds_wide.stack().rename(name)

meta_features = pd.concat(meta_frames, axis=1).sort_index()

# Next-hour return as the stacking target
meta_label = data["return"].shift(-1).stack().reindex(meta_features.index)

# Keep only rows where ALL base models AND the label are available
meta_mask = meta_features.notna().all(axis=1) & meta_label.notna()
meta_features = meta_features.loc[meta_mask]
meta_label = meta_label.loc[meta_mask]

print(f"Meta-feature matrix: {meta_features.shape}")
print(f"Base models:         {list(meta_features.columns)}")
print(f"Date range:          {meta_features.index.get_level_values(0).min()} → "
      f"{meta_features.index.get_level_values(0).max()}")

# ──────────────────────────────────────────────────────────────
# 2. Walk-forward stacking (monthly rebalancing, expanding window)
# ──────────────────────────────────────────────────────────────

RIDGE_ALPHA = 1.0  # regularization for the meta-learner

meta_time = meta_features.index.get_level_values(0)

# Monthly rebalancing dates within the validation period
meta_rebal_dates = pd.date_range(
    start=start_date_validate,
    end=last_date_validate,
    freq="ME",
)

stacked_preds_list = []

print(f"\nWalk-forward stacking with {len(meta_rebal_dates)} rebalancing dates\n")

for i, rebal_date in enumerate(meta_rebal_dates):

    # Meta-train: everything up to rebal_date (expanding window)
    meta_train_mask = meta_time <= rebal_date
    # Meta-predict: the month after rebal_date
    if i + 1 < len(meta_rebal_dates):
        next_date = meta_rebal_dates[i + 1]
    else:
        next_date = pd.to_datetime(last_date_validate)

    meta_pred_mask = (meta_time > rebal_date) & (meta_time <= next_date)

    X_meta_train = meta_features.loc[meta_train_mask].values
    y_meta_train = meta_label.loc[meta_train_mask].values
    X_meta_pred  = meta_features.loc[meta_pred_mask].values

    # Need enough data to fit
    if X_meta_train.shape[0] < 100 or X_meta_pred.shape[0] == 0:
        print(f"  [{rebal_date.date()}] skipped (train={X_meta_train.shape[0]}, "
              f"pred={X_meta_pred.shape[0]})")
        continue

    # Fit Ridge meta-learner on past OOS predictions
    ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    ridge.fit(X_meta_train, y_meta_train)

    # Predict the next month
    preds = ridge.predict(X_meta_pred).astype(np.float32)
    pred_series = pd.Series(preds, index=meta_features.loc[meta_pred_mask].index)
    stacked_preds_list.append(pred_series)

    # Report weights
    weights = dict(zip(meta_features.columns, ridge.coef_))
    w_str = ", ".join(f"{k}: {v:+.4f}" for k, v in weights.items())
    print(f"  [{rebal_date.date()}] train={X_meta_train.shape[0]:,}  "
          f"pred={X_meta_pred.shape[0]:,}  intercept={ridge.intercept_:.6f}")
    print(f"    weights → {w_str}")

# ──────────────────────────────────────────────────────────────
# 3. Aggregate stacked predictions
# ──────────────────────────────────────────────────────────────

predictions_stacked = (
    pd.concat(stacked_preds_list)
    .groupby(level=[0, 1])
    .first()   # no overlap expected, but safety
    .unstack()
    .sort_index()
)

print(f"\nStacked ensemble predictions shape: {predictions_stacked.shape}")
print(f"Validation coverage: "
      f"{predictions_stacked.dropna(how='all').index[0]} → "
      f"{predictions_stacked.dropna(how='all').index[-1]}")

predictions_stacked.dropna(how="all").head()

In [ ]:
# Add stacked ensemble to models dict
models["Stacked_Ensemble"] = predictions_stacked

# Analyze stacked ensemble performance
analyze_expected_returns(
    expected_returns=predictions_stacked.loc[start_date_validate:last_date_validate].dropna(how="all"),
    returns=data["return"].loc[start_date_validate:last_date_validate],
    rfr_hourly=rfr_hourly,
    title="Stacked Ensemble (Ridge meta-learner)",
    lags=[0, 1, 2, 3, 6, 12],
    tc=tc,
)

In [ ]:
# ── Compare all 5 models side-by-side ──

from support_functions import get_sharpe

def compute_finance_stats_v2(expected_returns, returns, rfr):
    """Compute key portfolio statistics for a model's expected returns."""
    positions = expected_returns_to_positions(expected_returns)
    pnl = positions.shift(1).mul(returns).sum(axis=1)

    sharpe = get_sharpe(pnl, rfr)
    ann_ret = pnl.mean() * 24 * 365
    ann_vol = pnl.std() * np.sqrt(24 * 365)
    cum_pnl = pnl.cumsum()
    max_dd = (cum_pnl - cum_pnl.cummax()).min()
    turnover = positions.fillna(0).diff().abs().sum(axis=1).mean() * 24

    return {
        "Sharpe": sharpe,
        "Ann.Return": f"{ann_ret:.4f}",
        "Ann.Vol": f"{ann_vol:.4f}",
        "Max.Drawdown": f"{max_dd:.4f}",
        "Turnover (daily)": f"{turnover:.2f}",
    }

stats_all = []
for name, preds in models.items():
    preds_val = preds.loc[start_date_validate:last_date_validate].dropna(how="all")
    stats_all.append({
        "Model": name,
        **compute_finance_stats_v2(preds_val, returns_val, rfr_hourly),
    })

stats_all_df = pd.DataFrame(stats_all).set_index("Model")
stats_all_df

In [ ]:
# ── Cumulative PnL: all models including stacked ensemble ──

plt.figure(figsize=(12, 5))

for name, preds in models.items():
    preds_val = preds.loc[start_date_validate:last_date_validate].dropna(how="all")
    positions = expected_returns_to_positions(preds_val)
    pnl = positions.shift(1).mul(returns_val).sum(axis=1)
    pnl.cumsum().plot(label=name, linewidth=1.5 if name == "Stacked_Ensemble" else 1.0)

plt.title("Cumulative PnL — All Models (Validation Period)")
plt.legend()
plt.grid(True)
plt.show()